In [26]:
import duckdb

con = duckdb.connect("db/unit-1-5.db")

In [23]:
con.close()

In [5]:
tables = con.sql("SHOW TABLES").df()
print(tables)

for table_name in tables["name"]:
    print(f"\nSchema for table: {table_name}")
    con.sql(f"DESCRIBE {table_name}").show()
    con.sql(f"SUMMARIZE {table_name}").show()

      name
0  address
1      car
2    claim
3   client

Schema for table: address
┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ id          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ country     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ state       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ city        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

┌─────────────┬─────────────┬───────────────┬───────────────┬───────────────┬─────────┬───────────────────┬─────────┬─────────┬─────────┬───────┬─────────────────┐
│ column_name │ column_type │      min      │      max      │ approx_unique │   avg   │        std        │   q25   │   q50

In [6]:
con.sql("SELECT \
  claim_date, claim_amt, \
  SUM(claim_amt) OVER (ORDER BY claim_date) AS running_total \
FROM claim")

┌─────────────────────┬───────────┬───────────────┐
│     claim_date      │ claim_amt │ running_total │
│       varchar       │   int64   │    int128     │
├─────────────────────┼───────────┼───────────────┤
│ 0001-01-18 00:00:00 │         0 │             1 │
│ 0001-01-18 00:00:00 │         1 │             1 │
│ 0001-01-20 00:00:00 │         0 │             1 │
│ 0001-02-18 00:00:00 │         0 │             1 │
│ 0001-03-18 00:00:00 │         3 │            11 │
│ 0001-03-18 00:00:00 │         7 │            11 │
│ 0001-03-19 00:00:00 │         0 │            11 │
│ 0001-03-20 00:00:00 │         0 │            17 │
│ 0001-03-20 00:00:00 │         6 │            17 │
│ 0001-05-20 00:00:00 │         3 │            20 │
│      ·              │         · │             · │
│      ·              │         · │             · │
│      ·              │         · │             · │
│ 9/3/20 0:00         │         0 │         19340 │
│ 9/4/18 0:00         │         0 │         19340 │
│ 9/4/19 0:0

In [8]:
con.sql("SELECT \
  cl.id,    \
  c.car_type,   \
  cl.claim_amt, \
  RANK() OVER ( \
    PARTITION BY   car_type \
    ORDER BY claim_amt DESC \
  ) AS rank \
FROM    \
  claim cl  \
  JOIN car c ON cl.car_id = c.id    \
QUALIFY rank = 1    \
ORDER BY    \
  cl.claim_amt DESC")

┌───────────┬─────────────┬───────────┬───────┐
│    id     │  car_type   │ claim_amt │ rank  │
│   int64   │   varchar   │   int64   │ int64 │
├───────────┼─────────────┼───────────┼───────┤
│ 515354210 │ Pickup      │       984 │     1 │
│ 673227790 │ SUV         │       942 │     1 │
│ 398144007 │ Minivan     │       909 │     1 │
│ 590103321 │ Sports Car  │       864 │     1 │
│  65426727 │ Van         │        49 │     1 │
│ 810479400 │ Panel Truck │        20 │     1 │
└───────────┴─────────────┴───────────┴───────┘

In [9]:
con.sql("WITH avg_claims AS ( \
  SELECT car_id, AVG(claim_amt) AS avg_claim_amt    \
  FROM claim    \
  GROUP BY car_id   \
)   \
SELECT car_id, avg_claim_amt    \
FROM avg_claims \
WHERE avg_claim_amt > ( \
  SELECT AVG(claim_amt) \
  FROM claim    \
)")

┌────────┬────────────────────┐
│ car_id │   avg_claim_amt    │
│ int64  │       double       │
├────────┼────────────────────┤
│     84 │               22.5 │
│    124 │               20.0 │
│    166 │               14.5 │
│    169 │              417.5 │
│    171 │               32.0 │
│    172 │ 232.66666666666666 │
│    176 │              432.0 │
│    238 │              228.5 │
│    276 │               17.0 │
│    302 │              328.0 │
│     ·  │                ·   │
│     ·  │                ·   │
│     ·  │                ·   │
│    776 │  281.6666666666667 │
│    785 │ 14.666666666666666 │
│    805 │              172.8 │
│    833 │              364.0 │
│    843 │              346.5 │
│    870 │               36.0 │
│    898 │              845.0 │
│    926 │              130.0 │
│    930 │              146.2 │
│    949 │              492.0 │
└────────┴────────────────────┘
  34 rows (20 shown)2 columns

In [11]:
con.sql("   \
WITH avg_claims AS (    \
  SELECT car_id, AVG(claim_amt) AS avg_claim_amt    \
  FROM claim    \
  GROUP BY car_id   \
),  \
overall_avg AS (    \
  SELECT AVG(claim_amt) AS overall_avg  \
  FROM claim    \
)   \
SELECT car_id, avg_claim_amt    \
FROM avg_claims \
WHERE avg_claim_amt > (SELECT overall_avg FROM overall_avg);    \
")

┌────────┬────────────────────┐
│ car_id │   avg_claim_amt    │
│ int64  │       double       │
├────────┼────────────────────┤
│     84 │               22.5 │
│    124 │               20.0 │
│    166 │               14.5 │
│    169 │              417.5 │
│    171 │               32.0 │
│    172 │ 232.66666666666666 │
│    176 │              432.0 │
│    238 │              228.5 │
│    276 │               17.0 │
│    302 │              328.0 │
│     ·  │                ·   │
│     ·  │                ·   │
│     ·  │                ·   │
│    776 │  281.6666666666667 │
│    785 │ 14.666666666666666 │
│    805 │              172.8 │
│    833 │              364.0 │
│    843 │              346.5 │
│    870 │               36.0 │
│    898 │              845.0 │
│    926 │              130.0 │
│    930 │              146.2 │
│    949 │              492.0 │
└────────┴────────────────────┘
  34 rows (20 shown)2 columns

In [30]:
## EXERCISE 5

"""
"""


"""
# lesson solution
con.sql(" \
WITH total_claims AS (  \
  SELECT car_id, SUM(claim_amt) AS total_claim_amt  \
  FROM claim    \
  GROUP BY car_id   \
)   \
SELECT car_id, total_claim_amt  \
FROM total_claims   \
WHERE total_claim_amt > (SELECT AVG(total_claim_amt) FROM total_claims) \
ORDER BY total_claim_amt DESC; \
")
"""

# my solution
con.sql("\
WITH total_claims AS (  \
	SELECT car_id, SUM(claim_amt) AS total_claim_amt    \
	FROM claim  \
	GROUP BY car_id \
),  \
overall_avg AS (    \
	SELECT AVG(claim_amt) AS overall_avg_amt    \
	FROM claim  \
)   \
SELECT car_id, total_claim_amt \
FROM total_claims   \
WHERE total_claim_amt > (  \
	SELECT overall_avg_amt FROM overall_avg)    \
ORDER BY total_claim_amt DESC; \
")


┌────────┬─────────────────┐
│ car_id │ total_claim_amt │
│ int64  │     int128      │
├────────┼─────────────────┤
│    949 │             984 │
│    302 │             984 │
│    445 │             942 │
│    434 │             934 │
│    238 │             914 │
│    736 │             875 │
│    805 │             864 │
│    176 │             864 │
│    776 │             845 │
│    898 │             845 │
│     ·  │               · │
│     ·  │               · │
│     ·  │               · │
│    975 │              19 │
│    600 │              18 │
│    993 │              18 │
│    527 │              17 │
│    594 │              16 │
│    920 │              15 │
│     36 │              15 │
│    764 │              14 │
│    291 │              14 │
│    257 │              14 │
└────────┴─────────────────┘
  54 rows        2 columns
  (20 shown)               